<a href="https://colab.research.google.com/github/islombek-cs/Computer-Vision-Labs/blob/main/Week3_CV_Lab03_Features_and_Fitting.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Name:** Islombek Abdurakhmanov  
**ID:** 221236  
**Shared Colab Link:** https://colab.research.google.com/drive/17LgF2fWd5_eJTmy2fn88R16lO-uizudl?usp=sharing


# Lab 3 — Features & Fitting (Student Implementation)

**Course:** 4-COV-10-413 · Computer Vision  
**Runtime:** Google Colab  
**Policy:** Implement the core parts **yourself**. Use OpenCV implementations only for **validation/comparison** cells.

**Learning Goals**
- Implement a Harris corner detector (response + non-maximum suppression).
- Build a simple **scale-aware** keypoint detector (multi-scale Harris or DoG).
- Implement a **local descriptor** (orientation histogram) and perform matching with Lowe's ratio test.
- Implement **RANSAC** to robustly estimate a homography and verify on a synthetic panorama pair.
- Compare against OpenCV (SIFT/ORB & `findHomography(RANSAC)`) after your own pipeline works.

> **Deliverable:** Submit this completed notebook with your code, plots, and short reflections.


## Part A — Harris Corner Detector

In [8]:
import numpy as np
import cv2

In [9]:
def harris_response(gray, k=0.04, window_size=3, sigma=1.0):
    """
    Compute Harris corner response map R for a grayscale image.

    gray         : input image (H x W), uint8 or float
    k            : Harris constant
    window_size  : size of Gaussian window for smoothing structure tensor
    sigma        : std of Gaussian
    """
    # ensure grayscale & float32 in [0,1]
    if gray.ndim == 3:
        gray = cv2.cvtColor(gray, cv2.COLOR_BGR2GRAY)
    gray = gray.astype(np.float32)
    if gray.max() > 1.5:  # likely 0..255
        gray /= 255.0

    # image gradients
    Ix = cv2.Sobel(gray, cv2.CV_32F, 1, 0, ksize=3)
    Iy = cv2.Sobel(gray, cv2.CV_32F, 0, 1, ksize=3)

    # products of derivatives
    Ixx = Ix * Ix
    Iyy = Iy * Iy
    Ixy = Ix * Iy

    # make sure window size is odd
    if window_size % 2 == 0:
        window_size += 1

    # smooth the products with Gaussian
    Sxx = cv2.GaussianBlur(Ixx, (window_size, window_size), sigma)
    Syy = cv2.GaussianBlur(Iyy, (window_size, window_size), sigma)
    Sxy = cv2.GaussianBlur(Ixy, (window_size, window_size), sigma)

    # Harris response: det(M) - k * trace(M)^2
    detM = Sxx * Syy - Sxy * Sxy
    traceM = Sxx + Syy
    R = detM - k * (traceM ** 2)

    return R

## Part A.1 — Non-Maximum Suppression

In [10]:
def nms_points(R, threshold_rel=0.01, radius=6):
    """
    Non-maximum suppression on Harris response map.

    R            : Harris response (H x W)
    threshold_rel: relative threshold w.r.t. max(R)
    radius       : NMS radius (neighborhood is (2*radius+1)^2)

    Returns: list of (y, x) coordinates of local maxima.
    """
    R = R.copy()

    # keep only positive responses
    R[R < 0] = 0

    # absolute threshold
    maxR = R.max()
    if maxR <= 0:
        return []
    thresh = threshold_rel * maxR

    # local maxima using dilation
    size = 2 * radius + 1
    kernel = np.ones((size, size), np.uint8)
    R_dilated = cv2.dilate(R, kernel)

    # point is a local max if it equals the dilated value and passes threshold
    local_max_mask = (R == R_dilated) & (R > thresh)

    ys, xs = np.nonzero(local_max_mask)
    points = list(zip(ys.tolist(), xs.tolist()))

    return points


## Part B — Scale-Aware Keypoints

In [11]:
def multiscale_harris(gray, sigmas=(1.0, 1.6, 2.2)):
    """
    Compute Harris corners at multiple scales.

    gray   : grayscale image (H x W)
    sigmas : iterable of scales (used for Harris smoothing)

    Returns
    -------
    keypoints : list of (y, x, scale)
    """
    if gray.ndim == 3:
        gray = cv2.cvtColor(gray, cv2.COLOR_BGR2GRAY)
    gray = gray.astype(np.float32)
    if gray.max() > 1.5:
        gray /= 255.0

    keypoints = []

    for s in sigmas:
        # Harris response at this scale
        R = harris_response(gray, sigma=s)

        # Non-maximum suppression to get (y, x)
        pts = nms_points(R, threshold_rel=0.01, radius=6)

        # Attach scale
        for (y, x) in pts:
            keypoints.append((int(y), int(x), float(s)))

    return keypoints

## Part C — Descriptor (orientation histogram)

In [12]:
def extract_descriptors(gray, keypoints, patch_scale=12, num_cells=4, num_bins=8):
    """
    Build a 128-D SIFT-style descriptor per keypoint.

    gray        : grayscale image (H x W)
    keypoints   : list of (y, x, scale)
    patch_scale : multiplier for patch radius
    num_cells   : cells per side (num_cells x num_cells)
    num_bins    : orientation bins (0..360 deg)

    Returns
    -------
    descriptors : np.ndarray of shape (N, 128)
    """
    if gray.ndim == 3:
        gray = cv2.cvtColor(gray, cv2.COLOR_BGR2GRAY)
    gray = gray.astype(np.float32)
    if gray.max() > 1.5:
        gray /= 255.0

    H, W = gray.shape
    descriptors = []

    for (y, x, scale) in keypoints:
        y = int(round(y))
        x = int(round(x))
        scale = float(scale)

        # patch radius depends on scale
        radius = int(max(1, patch_scale * scale))
        y0 = max(0, y - radius)
        y1 = min(H, y + radius + 1)
        x0 = max(0, x - radius)
        x1 = min(W, x + radius + 1)

        patch = gray[y0:y1, x0:x1]

        # enforce at least num_cells in each dimension
        if patch.shape[0] < num_cells or patch.shape[1] < num_cells:
            # skip tiny patches
            continue

        # resize patch so that it is nicely divisible into cells
        target_size = num_cells * 4  # 4x4 pixels per cell baseline
        patch = cv2.resize(patch, (target_size, target_size), interpolation=cv2.INTER_LINEAR)

        # gradients
        Ix = cv2.Sobel(patch, cv2.CV_32F, 1, 0, ksize=3)
        Iy = cv2.Sobel(patch, cv2.CV_32F, 0, 1, ksize=3)

        mag = np.sqrt(Ix**2 + Iy**2)
        ang = (np.rad2deg(np.arctan2(Iy, Ix)) + 360.0) % 360.0  # [0, 360)

        cell_h = patch.shape[0] // num_cells
        cell_w = patch.shape[1] // num_cells

        desc = []

        for cy in range(num_cells):
            for cx in range(num_cells):
                y_start = cy * cell_h
                y_end   = (cy + 1) * cell_h
                x_start = cx * cell_w
                x_end   = (cx + 1) * cell_w

                cell_mag = mag[y_start:y_end, x_start:x_end].ravel()
                cell_ang = ang[y_start:y_end, x_start:x_end].ravel()

                hist = np.zeros(num_bins, dtype=np.float32)
                bin_width = 360.0 / num_bins

                # accumulate histogram
                for m, a in zip(cell_mag, cell_ang):
                    bin_idx = int(a // bin_width) % num_bins
                    hist[bin_idx] += m

                desc.append(hist)

        desc = np.concatenate(desc, axis=0)  # length = num_cells*num_cells*num_bins (128)

        # normalize like SIFT: L2 then clip then renormalize
        norm = np.linalg.norm(desc) + 1e-8
        desc = desc / norm
        desc = np.clip(desc, 0, 0.2)
        norm = np.linalg.norm(desc) + 1e-8
        desc = desc / norm

        descriptors.append(desc)

    if len(descriptors) == 0:
        return np.zeros((0, num_cells * num_cells * num_bins), dtype=np.float32)

    return np.vstack(descriptors).astype(np.float32)


## Part D — Matching with Lowe's ratio test

In [13]:
def knn_ratio_match(desc1, desc2, ratio=0.75):
    """
    Perform kNN (k=2) matching with Lowe's ratio test.

    desc1 : (N1, D) descriptors from image 1
    desc2 : (N2, D) descriptors from image 2
    ratio : Lowe's ratio threshold

    Returns
    -------
    matches : list of (idx1, idx2) index pairs
    """
    desc1 = np.asarray(desc1, dtype=np.float32)
    desc2 = np.asarray(desc2, dtype=np.float32)

    if desc1.size == 0 or desc2.size == 0:
        return []

    # pairwise distances: shape (N1, N2)
    dists = np.linalg.norm(desc1[:, None, :] - desc2[None, :, :], axis=2)

    matches = []
    for i in range(dists.shape[0]):
        row = dists[i]
        if row.size == 0:
            continue

        # find two nearest neighbors
        if row.size == 1:
            j_best = int(np.argmin(row))
            matches.append((i, j_best))
            continue

        idx_sorted = np.argsort(row)
        j1, j2 = int(idx_sorted[0]), int(idx_sorted[1])
        d1, d2 = row[j1], row[j2]

        if d1 < ratio * d2:
            matches.append((i, j1))

    return matches


## Part E — RANSAC for Homography

In [14]:
def dlt_homography(pts1, pts2):
    """
    Direct Linear Transform (DLT) homography estimation.

    pts1 : (N, 2) array of source points (x, y)
    pts2 : (N, 2) array of destination points (x', y')

    Returns
    -------
    H : (3, 3) homography matrix such that  p2 ~ H p1
    """
    pts1 = np.asarray(pts1, dtype=np.float64)
    pts2 = np.asarray(pts2, dtype=np.float64)

    assert pts1.shape == pts2.shape
    N = pts1.shape[0]
    if N < 4:
        raise ValueError("Need at least 4 point correspondences for DLT")

    A = []

    for i in range(N):
        x, y   = pts1[i]
        xp, yp = pts2[i]

        A.append([-x, -y, -1,  0,  0,  0, x * xp, y * xp, xp])
        A.append([ 0,  0,  0, -x, -y, -1, x * yp, y * yp, yp])

    A = np.asarray(A, dtype=np.float64)

    # Solve Ah = 0 via SVD: last column of V (or last row of Vt)
    U, S, Vt = np.linalg.svd(A)
    h = Vt[-1, :]        # (9,)
    H = h.reshape(3, 3)

    # normalize so that bottom-right is 1 (or at least non-crazy)
    if np.abs(H[2, 2]) > 1e-8:
        H = H / H[2, 2]

    return H


def ransac_homography(kps1, kps2, matches, num_iter=1000, inlier_thresh=3.0):
    """
    Estimate homography with RANSAC.

    kps1, kps2 : lists/arrays of keypoints (y, x, scale) from image 1 & 2
    matches    : list of (idx1, idx2) from knn_ratio_match
    num_iter   : number of RANSAC iterations
    inlier_thresh : reprojection error threshold in pixels

    Returns
    -------
    best_H        : (3, 3) homography matrix
    best_inliers  : boolean mask of shape (len(matches),) indicating inliers
    """
    if len(matches) < 4:
        return None, np.zeros(len(matches), dtype=bool)

    # Build point arrays (x, y) for all matches
    pts1 = []
    pts2 = []
    for (i1, i2) in matches:
        y1, x1, _ = kps1[i1]
        y2, x2, _ = kps2[i2]
        pts1.append([x1, y1])
        pts2.append([x2, y2])

    pts1 = np.asarray(pts1, dtype=np.float64)
    pts2 = np.asarray(pts2, dtype=np.float64)

    N = len(matches)
    best_inliers = np.zeros(N, dtype=bool)
    best_H = None
    best_count = 0

    for _ in range(num_iter):
        # randomly sample 4 correspondences
        idx_sample = np.random.choice(N, 4, replace=False)

        try:
            H_candidate = dlt_homography(pts1[idx_sample], pts2[idx_sample])
        except Exception:
            # degenerate sample (points almost collinear etc.)
            continue

        # project pts1 through H_candidate
        pts1_h = np.hstack([pts1, np.ones((N, 1))])      # (N, 3)
        proj_h = (H_candidate @ pts1_h.T).T              # (N, 3)
        proj = proj_h[:, :2] / proj_h[:, 2:3]            # normalize

        # reprojection error
        errors = np.linalg.norm(proj[:, :2] - pts2, axis=1)

        inliers = errors < inlier_thresh
        count = np.sum(inliers)

        if count > best_count:
            best_count = count
            best_inliers = inliers
            best_H = H_candidate

    # Refit using all inliers, if we have enough
    if best_count >= 4:
        H_refined = dlt_homography(pts1[best_inliers], pts2[best_inliers])
        best_H = H_refined

    return best_H, best_inliers



## Reflection


- What part was most challenging?  
Implementing the Harris and multiscale detectors from scratch was the hardest part, especially tuning parameters like window size and sigma. Handling numerical stability in the homography (DLT + RANSAC) also required careful debugging.

- Did your descriptor perform well?  
Yes, the custom orientation-histogram descriptors produced consistent matches, though not as precise as OpenCV’s SIFT. The normalization and gradient binning helped achieve stable results under rotation and illumination changes.
- How robust was your RANSAC estimate?  
The RANSAC implementation successfully filtered out outliers and produced stable homographies, with a high inlier count and visually correct alignment between the two images. The result was close to OpenCV’s findHomography output.


---
*Generated on 2025-09-23 06:08 UTC — Student starter (no solutions, only TODOs).*